# Adaptive, Cost-Aware Multi-Agent Code Generation and Self-Debugging on Open-Weight Language Models
### Interactive Google Colab Replication & Verification Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rg0rajesh/Multi-Coding-Agent/blob/main/Multi_Coding_Agent_Replication.ipynb)

**Official GitHub Repository:** [https://github.com/Rg0rajesh/Multi-Coding-Agent](https://github.com/Rg0rajesh/Multi-Coding-Agent)  
**Primary Branch:** `main` | **Commit:** `c48a40c` | **License:** MIT Open-Source  
**Authors:** Rajesh et al.  

---

## Overview & Purpose
This interactive Google Colab notebook provides an end-to-end, reproducible environment for verifying all experimental results, benchmark tables, and multi-agent pipeline components from the research paper.

### Core Architectural Modules:
1. **Module 1 (Adaptive Threshold Controller):** Ranks error severity, detects stagnation and oscillation patterns ($A \to B \to A$), extends iteration budget only on progress, and manages score-based rollback.
2. **Module 2 (Test Oracle Generator):** Extracts zero-token unit test assertions from docstrings and backticks with an LLM fallback generator for test-free tasks.
3. **Module 3 (Hybrid Verification Gate):** Pre-screens syntax errors with AST parsing and executes candidates in an isolated subprocess sandbox with 5.0s timeout.

### Verifiable Research Results:
- **Table I:** Full Pipeline Pass@1 on HumanEval (71.95%–92.68%) and MBPP (50.10%–78.03%) across 6 open-weight models.
- **Table II:** Ablation study confirming an average **16.6% token reduction** via Module 1.
- **Table III:** Zero-Test diagnostic benchmark demonstrating **+4 to +16 pp Pass@1 gain** purely from Module 2 test synthesis.
- **Table IV:** Tiered Hybrid Verifier achieving **85%–90% agreement** with ground-truth correctness (Cohen's $\kappa = 0.70 - 0.80$).

## Section 1: Clone Repository & Setup Working Directory
Clones the verified research artifact repository directly from GitHub.

In [ ]:
# Clone repository and enter directory
import os
import sys

if not os.path.exists('Multi-Coding-Agent'):
    !git clone https://github.com/Rg0rajesh/Multi-Coding-Agent.git
    %cd Multi-Coding-Agent
else:
    %cd Multi-Coding-Agent
    !git pull origin main

# Verify git log and working tree
!git status
!git log -1 --oneline

## Section 2: Install Dependencies
Installs the required Python dependencies specified in `requirements.txt`.

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

## Section 3: Master Automated Verification Suite (Zero-Dependency Audit)
Executes `verify_all_results.py` to audit 100% of all JSONL experimental execution logs against published tables in the paper:
- **Check 1/4:** MBPP 974 Deduplicated Tasks across 6 models
- **Check 2/4:** HumanEval 164 Tasks Full Pipeline across 6 models
- **Check 3/4:** Zero-Test 100-Task Diagnostic Suite across 6 models
- **Check 4/4:** Module 3 Tiered Verifier Reliability across 6 models

In [ ]:
# Run zero-dependency verification audit
!python verify_all_results.py

## Section 4: Dynamic Paper Table Reproduction
Executes `reproduce_paper_tables.py` to dynamically compute Table I (Pass@1, Tokens, Latency) and Table IX (Iteration Convergence Distributions) directly from raw JSONL traces.

In [ ]:
# Reproduce benchmark tables
!python reproduce_paper_tables.py

## Section 5: Statistical Significance & Paired McNemar Tests
Runs paired statistical comparisons assessing whether Module 2's oracle synthesis and Module 3's hybrid verification are statistically significant ($p < 0.05$).

In [ ]:
# Run paired comparison and significance tests
!python -m analysis.paired_comparison
!python -m analysis.significance_test

## Section 6: Results Visualization (Pass@1 & Token Consumption)
Generates visual comparison figures for the research paper results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load summary data
df = pd.read_csv('results/summaries/full_ablation_summary.csv')

models = [
    ('qwen2.5-coder:0.5b', 'Qwen 0.5B'),
    ('qwen2.5-coder:1.5b', 'Qwen 1.5B'),
    ('deepseek-coder:1.3b', 'DS 1.3B'),
    ('llama3.2:3b', 'Llama 3B'),
    ('qwen2.5-coder:3b', 'Qwen 3B'),
    ('deepseek-coder:6.7b', 'DS 6.7B'),
]

base_pass, pipe_pass = [], []
base_tok, pipe_tok = [], []

for slug, name in models:
    r_base = df[(df['model'] == slug) & (df['config'] == 'baseline')]
    r_pipe = df[(df['model'] == slug) & (df['config'] == '+module1+module2+module3') & (df['n'] == 164)]
    base_pass.append(r_base['pass_at_1'].values[0] * 100)
    pipe_pass.append(r_pipe['pass_at_1'].values[0] * 100)
    base_tok.append(r_base['avg_tokens'].values[0])
    pipe_tok.append(r_pipe['avg_tokens'].values[0])

x = np.arange(len(models))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Pass@1
ax1.bar(x - width/2, base_pass, width, label='Baseline Repair', color='#333333', edgecolor='black')
ax1.bar(x + width/2, pipe_pass, width, label='Proposed Pipeline', color='#888888', edgecolor='black')
ax1.set_ylabel('Pass@1 Accuracy (%)', fontsize=11, fontweight='bold')
ax1.set_title('HumanEval Pass@1: Baseline vs Full Pipeline', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([m[1] for m in models], fontsize=10)
ax1.set_ylim(0, 100)
ax1.legend(loc='lower right')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Subplot 2: Token Reduction
reduction_pct = [((b - p) / b) * 100 for b, p in zip(base_tok, pipe_tok)]
bars = ax2.bar(x, reduction_pct, width=0.45, color='#222222', edgecolor='black')
ax2.set_ylabel('Token Reduction (%)', fontsize=11, fontweight='bold')
ax2.set_title('Token Consumption Reduction (HumanEval Average: 16.6%)', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([m[1] for m in models], fontsize=10)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    h = bar.get_height()
    ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('colab_benchmark_visualization.png', dpi=300)
plt.show()
print('Saved plot to colab_benchmark_visualization.png')

## Section 7: Live Multi-Agent Component Test (Interactive)
Executes an interactive test of Module 2 (Test Oracle Extraction) and Module 3 (Sandbox Execution) on a sample Python programming problem.

In [ ]:
from my_pipeline.module2_test_oracle import Module2TestOracleGenerator
from my_pipeline.sandbox_exec import SandboxExecutor

# Sample task prompt with embedded doctest examples
sample_prompt = '''from typing import List

def filter_by_prefix(strings: List[str], prefix: str) -> List[str]:
    """ Filter an input list of strings only for state starting with a given prefix.
    >>> filter_by_prefix([], 'a')
    []
    >>> filter_by_prefix(['abc', 'bcd', 'cde', 'array'], 'a')
    ['abc', 'array']
    """
'''

# 1. Module 2: Zero-Token Test Oracle Extraction
oracles = Module2TestOracleGenerator.extract_zero_token_oracles(sample_prompt, 'filter_by_prefix')
print("=== MODULE 2: EXTRACTED TEST ORACLES (Zero Token Cost) ===")
for idx, o in enumerate(oracles, start=1):
    print(f"{idx}. {o}")

# 2. Candidate Solution to Verify
candidate_code = """from typing import List

def filter_by_prefix(strings: List[str], prefix: str) -> List[str]:
    return [x for x in strings if x.startswith(prefix)]
"""

# 3. Module 3: Sandbox Verification
test_harness = candidate_code + "\n" + "\n".join(oracles) + "\nprint('ALL EXTRACTED ORACLES PASSED IN SANDBOX!')"
result = SandboxExecutor.run_code(test_harness, timeout_sec=5.0)

print("\n=== MODULE 3: SANDBOX VERIFICATION RESULT ===")
print(f"Verification Status: {'PASSED' if result.passed else 'FAILED'}")
print(f"Execution Output:    {result.stdout.strip()}")
print(f"Execution Latency:   {result.time_sec:.4f} seconds")

## Section 8: Conclusion & Citation Information

### Summary of Verification:
1. All six models are mathematically validated against the raw JSONL traces (0 discrepancies).
2. Module 1 reliably terminates unproductive loops, yielding 16.6% average token savings.
3. Module 2 produces valid test assertions directly from docstrings, boosting Pass@1 by +4 to +16 pp when tests are absent.
4. Module 3 achieves 85%–90% agreement with ground-truth correctness.

```bibtex
@article{rajesh2026multicodingagent,
  title={Adaptive, Cost-Aware Multi-Agent Code Generation and Self-Debugging on Open-Weight Language Models},
  author={Rajesh et al.},
  journal={Under Peer Review},
  year={2026}
}
```